# 完整圖片推論

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/07-inference/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
import torch
from miniyolo.data import collate
from miniyolo.models import GridDetector
from miniyolo.targets import build_targets
from miniyolo.inference import decode_grid


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    image = torch.zeros(3, 64, 64)
    image[0, 12:28, 8:24] = 1
    image[2, 36:52, 40:56] = 1
    two = {'boxes': torch.tensor([[8.,12.,24.,28.], [40.,36.,56.,52.]]),
           'labels': torch.tensor([0,1])}
    one = {'boxes': two['boxes'][:1], 'labels': two['labels'][:1]}
    empty = {'boxes': torch.empty(0,4), 'labels': torch.empty(0,dtype=torch.long)}
    one_image = image.clone()
    one_image[2,36:52,40:56] = 0  # The one-object image really has no blue pixels.
    assert one_image[2].sum() == 0 and one_image[0].sum() == 256
    images, anns = collate([(image,two), (one_image,one), (torch.zeros_like(image),empty)])
    model = GridDetector(num_classes=2, grid_size=4, width=8).eval()
    with torch.inference_mode():
        raw = model(images)
        smoke = decode_grid(raw, image_size=64, score_threshold=.25, nms_iou=.5)
    assert len(smoke) == 3
    assert all(p['boxes'].shape == (len(p['scores']),4) for p in smoke)
    print('untrained model candidate counts (no quality claim)', [len(p['boxes']) for p in smoke])
    target = build_targets(anns,4,64,2)
    fixture = torch.zeros_like(raw)
    fixture[...,4] = -20
    for b,y,x in target['positive'].nonzero().tolist():
        fixture[b,y,x,:4] = torch.logit(target['box'][b,y,x].clamp(1e-4,1-1e-4))
        fixture[b,y,x,4] = 10
        fixture[b,y,x,5:] = -10
        fixture[b,y,x,5+target['class_ids'][b,y,x].item()] = 10
    decoded = decode_grid(fixture,64,.25,.5)
    assert [len(p['boxes']) for p in decoded] == [2,1,0]
    for predicted, ann in zip(decoded, anns):
        # Tied scores may change ordering; compare each class separately.
        for cls in ann['labels'].unique().tolist():
            assert torch.allclose(predicted['boxes'][predicted['labels']==cls],
                                  ann['boxes'][ann['labels']==cls], atol=.02)
    assert len(decode_grid(fixture[:1],64,.25,.5)) == 1
    print('artificial known-logit fixture counts', [len(p['boxes']) for p in decoded])
    print('fixture red box', decoded[1]['boxes'][0].tolist())
    duplicate = fixture[1:2].clone()
    duplicate[0,1,0,:4] = torch.logit(torch.tensor([.999,.25,.25,.25]))
    duplicate[0,1,0,4:] = torch.tensor([9.,10.,-10.])
    # The neighbor cell predicts almost the same red box, with a lower score.
    before_nms = decode_grid(duplicate,64,.25,1.0)[0]
    after_nms = decode_grid(duplicate,64,.25,.5)[0]
    assert len(before_nms['boxes']) == 2 and len(after_nms['boxes']) == 1
    print('same-class overlapping fixture NMS counts',2,'->',1)


if __name__ == '__main__':
    main()


untrained model candidate counts (no quality claim) [0, 0, 0]
artificial known-logit fixture counts [2, 1, 0]
fixture red box [8.00160026550293, 12.0, 24.00160026550293, 28.0]
same-class overlapping fixture NMS counts 2 -> 1
